In [ ]:
import os
import pandas as pd
import seaborn as sns
import plotly.express as px
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
train_df = pd.read_csv("data/HAM10000_metadata")
test_df = pd.read_csv("data/ISIC2018_Task3_Test_GroundTruth.csv")

def make_label(dx):
    if dx == "nv":
        return "nv"
    elif dx == "mel":
        return "mel"
    else:
        return "others"

train_df["label_3class"] = train_df["dx"].apply(make_label)
test_df["label_3class"] = test_df["dx"].apply(make_label)

def find_image_path(image_id, folders):
    for folder in folders:
        path = os.path.join(folder, f"{image_id}.jpg")
        if os.path.exists(path):
            return path
    return None

train_folders = [
    "data/HAM10000_images_part_1",
    "data/HAM10000_images_part_2"
]
test_folder = "data/ISIC2018_Task3_Test_Images"

train_df["path"] = train_df["image_id"].apply(lambda x: find_image_path(x, train_folders))
test_df["path"] = test_df["image_id"].apply(lambda x: find_image_path(x, [test_folder]))

In [ ]:
px.histogram(train_df, x="label_3class", title="Distribution of 3-class labels in training set")

In [ ]:
px.histogram(test_df, x="label_3class", title="Distribution of 3-class labels in testing set")

In [ ]:
for label in train_df["label_3class"].unique():
    sample = train_df[train_df["label_3class"] == label].sample(3, random_state=42)
    fig, axes = plt.subplots(1, 3, figsize=(20, 5))
    for i, (_, row) in enumerate(sample.iterrows()):
        img = Image.open(row["path"])
        axes[i].imshow(img)
        axes[i].set_title(f"{row['path'].split('/')[-1]} ({row['dx']})")
        axes[i].axis("off")
    plt.suptitle(f"Sample images for label: {label}")
    plt.show()

In [ ]:
train_final = train_df[["image_id", "label_3class", "path"]]
test_final = test_df[["image_id", "label_3class", "path"]]
train_final.to_csv("train_data.csv", index=False)
test_final.to_csv("test_data.csv", index=False)